# 20 — Tool Calling, Function Calling, and Structured Actions

**Network LLM Engineering — Part V — Agentic Systems**

### Learning goals
- Understand tool schemas and model/tool boundaries
- Build a safe mock networking tool
- Prevent the LLM from inventing live state

## The key rule

If the answer depends on **current network state**, call a tool.
Do not ask the model to "remember" whether Router-17 is currently Established with a peer.

A tool schema tells the model what capability exists and what arguments it accepts.
The application—not the model—executes the tool.

In [ ]:
import json
MOCK_STATE = {
    "R1": {
        "bgp_neighbors": {"192.0.2.2":"Established"},
        "interfaces": {"Gi0/0":{"state":"up","crc":0}, "Gi0/1":{"state":"up","crc":432}},
    }
}

def get_bgp_neighbors(device):
    if device not in MOCK_STATE:
        raise KeyError("unknown device")
    return MOCK_STATE[device]["bgp_neighbors"]

def get_interface(device, interface):
    return MOCK_STATE[device]["interfaces"][interface]

print(get_bgp_neighbors("R1"))
print(get_interface("R1","Gi0/1"))

## Separation of responsibilities

**LLM:** decides which evidence may be useful; interprets human language; synthesizes.
**Tool:** returns authoritative current data.
**Validator/policy:** checks arguments and permissions.
**Orchestrator:** controls retries, timeouts, state, and next steps.

This is more reliable than giving the LLM direct unrestricted shell access.

In [ ]:
tool_schema = {
  "name":"get_interface",
  "description":"Read current interface state and counters",
  "parameters":{
    "type":"object",
    "properties":{
      "device":{"type":"string"},
      "interface":{"type":"string"}
    },
    "required":["device","interface"]
  }
}
print(json.dumps(tool_schema, indent=2))

### Exercise

Add a read-only `get_route(device, prefix)` tool.
Then design a **separate** privileged change tool with stricter authorization and human approval.